# 43 · Evaluation-bundle export and self-contained loading

Package the explicitly selected fold-01 research checkpoint. This does not select a production model or run training.
The exporter removes training-only fields and local paths. It preserves all retained values and tensor bits.
It verifies every selected validation epoch against saved predictions before publishing four immutable bundle files.
The loader uses only those files and the matching installed runtime. It does not need the original artifact tree or motion cache.
The final-test participants remain reserved.

Source-checkout export and loading require clean runtime inputs at the bundle's exact source revision.
Commit runtime changes before exporting. A different commit requires a new bundle identifier and matching runtime.
Documentation comes from the recorded commit. Commit model-card edits before packaging them.
Read the [operating guide](../docs/evaluation-bundles.md).


## Locate the checkout and opt-in operations

**What:** Resolve the checkout and declare optional packaging and inference.

**Why:** A report-only run must not export files, run inference, or train a model.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("evaluation_bundle_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/evaluation-bundle.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "feature/evaluation-bundles"  # Select the PR branch before merge
RUN_SETUP = False
RUN_EXPORT = False
RUN_INFERENCE = False

logger.info("Repository: %s", ROOT)
logger.info("Setup=%s export=%s inference=%s", RUN_SETUP, RUN_EXPORT, RUN_INFERENCE)

## Optional environment setup

**What:** Clone the PR branch and install the locked runtime and notebook groups.

**Why:** A new environment needs the same package versions; existing optional groups stay installed.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --inexact --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")

## Verify the explicit source and runtime

**What:** Inspect the selected source fold, passed audit, runtime revision, and immutable destinations.

**Why:** Exports must trace back to the measured checkpoint and the exact code that can load them.


In [ ]:
import pandas as pd

from sleep_classifier.bundle_export import EvaluationBundleExporter

exporter = EvaluationBundleExporter(ROOT / "configs/evaluation-bundle.yaml")
plan = exporter.plan()
logger.info("Export config=%s output=%s fold=%s revision=%s", exporter.config_path,
            exporter.output_dir, plan["fold"], plan["runtime"]["source_revision"])
display(pd.DataFrame([plan["runtime"]]))
display(pd.DataFrame([plan["source"]["fold"]]).drop(columns=["features"]))
logger.info("New exports require MPS. Inspection does not select another backend.")

## Export only after explicit opt-in

**What:** Build a sanitized bundle and require full selected-fold prediction parity before publication.

**Why:** No source checkpoint is changed, and an existing identifier cannot be overwritten.


In [ ]:
if RUN_EXPORT:
    if exporter.output_dir.exists():
        exporter.read_existing()
        logger.info("Verified existing immutable bundle; export was not repeated")
    else:
        export_config = exporter.config_path
        exporter.validate_start()
        logger.info("Export config=%s output=%s fold=%s", export_config,
                    exporter.output_dir, exporter.settings.fold)
        %cd "{ROOT}"
        !uv run --locked --inexact --group gru python -u -m sleep_classifier.bundle_export \
            --config "{export_config}" --export
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Bundle export failed; retain source and partial evidence")
else:
    logger.info("Export skipped; no training or packaging command ran")

## Inspect file integrity and provenance

**What:** Verify the bundle schema, lock metadata, layout, and file hashes without restoring model weights.

**Why:** Results remain inspectable after a documentation commit changes the checkout revision.
This inspection does not validate checkpoint semantics or approve inference under a different runtime.


In [ ]:
from sleep_classifier.bundle_loader import inspect_bundle

manifest = None

if exporter.output_dir.exists():
    manifest = inspect_bundle(exporter.output_dir)
    display(pd.DataFrame([{"role": role, **record} for role, record in manifest["files"].items()])
            .set_index("role"))
    display(pd.DataFrame([manifest["provenance"]]))
    recorded_revision = manifest["runtime"]["source_revision"]
    current_revision = plan["runtime"]["source_revision"]
    logger.info("File integrity passed. Bundle role=%s recorded_revision=%s current_revision=%s",
                manifest["artifact_role"], recorded_revision, current_revision)
    if recorded_revision != current_revision:
        logger.info("Inference needs the matching runtime; inspection does not load weights")
else:
    logger.info("Bundle pending; enable RUN_EXPORT when ready")


## Optional synthetic request inference

**What:** Classify one synthetic, label-free epoch through the verified bundle.

**Why:** This demonstrates the JSON interface without reading any sensor or final-test observations.


In [ ]:
from sleep_classifier.bundle_validation import strict_json_loads

if RUN_INFERENCE:
    if manifest is None:
        raise FileNotFoundError("Export or restore the configured bundle before inference")
    bundle = exporter.read_existing()
    logger.info("Saved motion imputation mean=%s", bundle.motion_imputation_mean)
    request_path = ROOT / "examples/inference-bundle/epoch-request.json"
    request = strict_json_loads(request_path.read_bytes())
    stream = bundle.predictor()
    response = stream.predict(request)
    display(response)
    display(stream.integrity())
    stream.reset()
else:
    logger.info("Inference skipped; no model forward pass ran")

## Interpretation and boundaries

The bundle remains an evaluation-fold artifact. It is not a deployment model or a calibrated clinical instrument.
Full export parity compares prepared validation summaries with saved probabilities at tolerance 2e-6 and zero class disagreements.
This adds packaging evidence, not a new accuracy result. Raw sensor ingestion remains outside this interface.

Loading requires the recorded source revision and compatible Python. Installed wheels contain their runtime identity and schemas.
A source checkout additionally requires clean runtime files. Do not bypass those checks after changing commits.
Keep immutable identifiers and their matching runtimes for rollback. Reset stream history when switching bundles.
The [contract](../docs/inference-bundle-contract.md) describes support rules and imputation requirements.

### Observed operator run · 2026-11-24

Export and loading passed at runtime revision `1c223c26` for the explicitly selected fold 01.
All 4,587 validation epochs preserved their predicted classes. The maximum probability difference was 7.45e-7, below tolerance 2e-6.
All history lengths and reset reasons matched. The model weights stayed unchanged, and no gradients were created.
The synthetic request returned a valid response; its predicted stage does not measure accuracy.
A separate matching-wheel check preserved all 33 synthetic responses with original source, data, artifact, and Git access blocked.

The saved bundle and wheel remain paired with that revision. Report inspection verifies their file integrity after later commits.
Model loading still requires the exact recorded runtime. Do not enable inference from an unmatched checkout.
See [measured evidence](../docs/experiments/evaluation-bundle-2026-11-24/README.md).


## Export the inspection report

**What:** Run report-only cells in a fresh kernel and save HTML.

**Why:** Report export skips optional packaging, inference, setup, and recursive export.


In [ ]:
%cd "{ROOT}"
!uv run --locked --inexact --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/43_evaluation_bundle.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Evaluation-bundle notebook export failed")